# Search and Deep Research: Finding Answers in Candidate Space


> Lecture 6 showed that a model can get stronger without changing weights. Lecture 7 let an Agent evolve itself in an open environment. Both lectures moved compute into the **inference** stage, but both skipped one step: after generating a large set of candidates, picking the one that actually works under a limited budget.
>
> This lecture turns "picking" into a full pipeline: first implement AlphaCode's sample, filter, and cluster steps in program space; then add a reranker that cuts the required sample count by orders of magnitude; then move into knowledge space and implement Search-o1's on-demand retrieval; finally chain the pieces into a small **deep research** workflow.

The answer to a programming contest problem is a program that passes every test; writing it correctly in one shot often falls short. AlphaCode's method is a pipeline that compresses a huge candidate set down to the 10 that can be submitted.

Step 1: sample 1 million candidate programs from the model. The volume is large; most of them cannot be used.

Step 2: run each candidate on a cheap example test and drop those that fail. 1 million is reduced to a few thousand.

Step 3: cluster the remaining few thousand by "whether behavior is the same"—candidates that produce the same outputs form one cluster, meaning they are essentially the same solution.

Step 4: the contest allows only 10 submissions, so pick one representative from each of a few dozen clusters, and submit exactly 10.

That is how a huge candidate set is picked down to something usable. This whole generate-drop-pick action is what the course calls **search**: having the model answer several times is sampling, dropping the unqualified is filtering, merging identical behavior is clustering. After the earlier lectures' "answer several times", this lecture adds one more step: "pick".

By the end of this lecture we can implement the core of three systems: AlphaCode's sample-and-filter pipeline, Search-o1's on-demand **retrieval** loop, and a deep research workflow that chains them. Placed in lecture 1's Agent loop, this lecture fills the step after "act"—after actions produce many results, deciding which one is worth submitting.

This section solves one problem: given a programming task and a batch of candidate programs, pick the one that will pass judging. We shrink the problem to one toy task and walk through sample, filter, and cluster; each step is computed by hand before code, so it is clear what each step drops and on what basis.

The toy programming task: write a function that returns True if the input integer is prime and False otherwise. Judging has two layers. The problem statement gives a few input-output pairs; these are the visible examples, called **example tests**. The submission system judges with another set of input-output pairs, called **hidden tests**, invisible for the entire run.

**Experimental corpus**: to simulate "the model sampled a batch of candidate programs", we prepare a candidate pool. The pool is scripted: three behaviorally correct implementations (different writing), plus six deliberately wrong variants, including treating 2 as composite, treating 1 as prime, and an off-by-one check range. The pipeline's only visible correctness signal is the example tests; hidden tests stay invisible throughout.

The shape of the whole pipeline can be seen in advance:

sample → filter → cluster → submit

At each station the candidate count drops by about an order of magnitude: a real system samples millions of programs from the model, example tests leave a few thousand, clustering compresses them into a few dozen clusters, and finally only 10 representatives are submitted under the quota. Each station uses a different basis: sampling relies on model ability, filtering on example tests as a cheap signal, clustering on whether candidates' output behavior matches, and submission on the quota.

The four stages are implemented one by one, each computed by hand before code. Once it is clear what each step drops and on what basis, the design of the pipeline is clear.

In [ ]:
import numpy as np

np.random.seed(42)

# Candidate program pool. In a real pipeline these are source strings sampled from a model, compiled with exec;
# for teaching clarity, candidates are represented directly as function objects.
def correct_naive(n):
    """Naive primality test: trial division from 2 through n-1."""
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def correct_sqrt(n):
    """Primality test: check only up to sqrt(n)."""
    if n < 2:
        return False
    i = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += 1
    return True


def correct_div2(n):
    """Primality test: handle evens first, then check odds stepping from 3."""
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    for i in range(3, n, 2):
        if n % i == 0:
            return False
    return True


def wrong_even_composite(n):
    """Bug: treat every even number as composite, so n=2 is misclassified."""
    if n < 2:
        return False
    if n % 2 == 0:
        return False
    for i in range(3, n):
        if n % i == 0:
            return False
    return True


def wrong_one_is_prime(n):
    """Bug: treat n=1 as prime."""
    if n == 1:
        return True
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def wrong_all_composite(n):
    """Bug: start checking at 1; n % 1 == 0 always, so everything is composite."""
    for i in range(1, n):
        if n % i == 0:
            return False
    return True


def wrong_trial_range(n):
    """Bug: check range misses the upper bound, so composites such as n=4 slip through."""
    for i in range(2, n // 2):
        if n % i == 0:
            return False
    return True


def wrong_mod_division(n):
    """Bug: modulo written as integer division; n // i == 0 never holds, so everything is prime."""
    for i in range(2, n):
        if n // i == 0:
            return False
    return True


def wrong_stop_early(n):
    """Bug: loop upper bound missing an equality, so perfect squares are misclassified."""
    i = 2
    while i * i < n:
        if n % i == 0:
            return False
        i += 1
    return True


POOL = [
    {"name": "correct_naive", "fn": correct_naive, "good": True},
    {"name": "correct_sqrt", "fn": correct_sqrt, "good": True},
    {"name": "correct_div2", "fn": correct_div2, "good": True},
    {"name": "wrong_even_composite", "fn": wrong_even_composite, "good": False},
    {"name": "wrong_one_is_prime", "fn": wrong_one_is_prime, "good": False},
    {"name": "wrong_all_composite", "fn": wrong_all_composite, "good": False},
    {"name": "wrong_trial_range", "fn": wrong_trial_range, "good": False},
    {"name": "wrong_mod_division", "fn": wrong_mod_division, "good": False},
    {"name": "wrong_stop_early", "fn": wrong_stop_early, "good": False},
]
POOL_BY_NAME = {p["name"]: p for p in POOL}

# Example tests: input-output pairs from the problem statement; the only correctness signal visible at filter time
EXAMPLE_TESTS = [(3, True), (4, False), (9, False), (13, True)]
# Hidden tests: revealed only at submission; invisible while the pipeline runs
HIDDEN_TESTS = [
    (1, False), (2, True), (5, True), (8, False), (11, True),
    (12, False), (17, True), (21, False), (23, True), (29, True),
]

n_good = sum(p["good"] for p in POOL)
print("candidate pool: %d programs, %d correct, %d incorrect"
      % (len(POOL), n_good, len(POOL) - n_good))
for p in POOL:
    print("  %-22s good=%s" % (p["name"], p["good"]))


The pool has 9 programs, 3 correct and 6 incorrect. To see what the filter stage drops and what it misses, we run each program on the 4 example-test inputs. A program's behavior is the map from input to output; here we only care whether it returns True or False on each input.

First (3, True). correct_naive trials from 2 to 2, finds 3 % 2 = 1, no exact division, returns True. wrong_all_composite starts at 1, 3 % 1 = 0, immediately returns False, and is filtered out at this step.

Next (4, False). correct_naive finds 4 % 2 = 0 at i = 2, returns False. wrong_trial_range only checks range(2, 2), an empty loop, and returns True. wrong_mod_division uses integer division n // i, the quotient is never 0, returns True. wrong_stop_early's loop condition is i*i < n; at i = 2, 4 < 4 is false, so it also returns True. All three variants are caught on this problem.

After all 4 example inputs, the full result table:

| Candidate | 3 | 4 | 9 | 13 | All passed |
|:---|:---|:---|:---|:---|:---|
| correct_naive | T | F | F | T | yes |
| correct_sqrt | T | F | F | T | yes |
| correct_div2 | T | F | F | T | yes |
| wrong_even_composite | T | F | F | T | yes |
| wrong_one_is_prime | T | F | F | T | yes |
| wrong_all_composite | F | F | F | F | no |
| wrong_trial_range | T | T | F | T | no |
| wrong_mod_division | T | T | T | T | no |
| wrong_stop_early | T | T | T | T | no |

T means the program returned True on that input, F means False. Four wrong variants are caught by example tests, but wrong_even_composite and wrong_one_is_prime pass all of them. Their defects fall on inputs the example tests do not cover: wrong_even_composite treats 2 as composite, wrong_one_is_prime treats 1 as prime, and the example tests happen not to include n = 1 and n = 2. They pass every example test, but their behavior differs from a correct solution: the same on covered inputs, different on uncovered ones. Hidden tests include (1, False) and (2, True); these two variants will fail on those inputs at submission.

Clustering needs an automatically computable rule for "which candidates can be treated as the same": run each candidate on a fixed set of inputs, record whether each input returns True or False, and obtain a 0/1 string. That string records a program's full behavior on those inputs; we call it a **behavior signature**. Programs with the same signature have the same behavior and are placed in the same cluster.

Compute one pass by hand. Take three candidates, run them on three probe inputs [1, 2, 9], and write the outputs as a row of 0/1 vectors.

In [ ]:
# Hand-calculated behavior signatures: 3 candidates × 3 probe inputs
probe = [1, 2, 9]
picks = ["correct_sqrt", "wrong_even_composite", "wrong_one_is_prime"]
mat = np.array([[1 if POOL_BY_NAME[n]["fn"](x) else 0 for x in probe]
                for n in picks])
print("probe inputs:", probe)
print("each row is one program's behavior signature (1 = classified as prime):")
for i, n in enumerate(picks):
    print("  %-20s %s" % (n, mat[i].tolist()))
print("Key observation: programs with different signatures fall in different clusters; programs with the same signature merge into one cluster.")


The three hand-computed signatures differ: correct_sqrt treats 1 and 9 as composite and 2 as prime, signature [0, 1, 0]; wrong_even_composite treats the prime 2 as composite, signature [0, 0, 0]; wrong_one_is_prime treats the composite 1 as prime, signature [1, 1, 0]. The signature separates the two "passed every example test" wrong variants from each other, and from the correct solution.

A signature is a program's full output record on a set of probe inputs. The same signature means that, within the inputs we care about, the two programs are indistinguishable; different signatures mean there is at least one input on which they give opposite results. Clustering merges candidates with the same signature into a group and keeps one representative per group.

Clustering can replace picking by hand because it turns the criterion into an automatically computable signal. The submission quota is only a few, while candidates may number in the thousands; reading each program to judge right or wrong is neither feasible nor reliable. Behavioral equivalence is that automatic signal: correct solutions output the right answer on any input, so their behavior matches and they form one large cluster; two wrong variants fail in different places, so they differ on probe inputs and form their own clusters. At submission we take representatives starting from the largest cluster; one representative covers the submission slot of every candidate in that cluster.

The first step is sampling: having the model generate many candidates. A real system samples millions of candidates from the model at a relatively high temperature at inference time; temperature controls how random the answers are—higher temperature yields more different answers to the same problem. Here a sampler stands in: given a quality parameter p_correct, each draw takes some correct implementation with that probability, otherwise one of the wrong variants. The quality parameter is a proxy for model ability: a stronger model has a higher probability of sampling a correct implementation.

In [ ]:
def sample_candidates(rng, n, p_correct=0.2):
    """Simulate LLM sampling: take a correct program with probability p_correct, otherwise a wrong one.

    n is the number of samples; quality parameter p_correct is a proxy for model ability; repeats are allowed.
    """
    good = [p for p in POOL if p["good"]]
    bad = [p for p in POOL if not p["good"]]
    picked = []
    for _ in range(n):
        if rng.random() < p_correct:
            picked.append(rng.choice(good))
        else:
            picked.append(rng.choice(bad))
    return picked


rng = np.random.default_rng(42)
N = 300
cands = sample_candidates(rng, N, p_correct=0.2)
n_ok = sum(c["good"] for c in cands)
print("sampled %d candidates, of which %d are correct (%.1f%%)"
      % (N, n_ok, 100.0 * n_ok / N))


The sampler compresses model ability into one parameter p_correct: each independent draw takes a correct implementation with probability p_correct and a wrong variant with probability 1 - p_correct. It replaces real model inference with reproducible, controllable results.

Each sample is correct with probability p_correct, so the expected number correct in n samples is n × p_correct. 300 × 0.2 = 60; the run actually drew 61, matching the expectation, which is the direct consequence of "independent draws, constant probability". In the AlphaCode paper the model's per-sample correctness is far below 0.2, roughly between 0.1% and 1%, so millions of candidates are sampled to guarantee that at least some correct solutions appear.

The second step is filtering. The verifier runs candidates on example tests and drops any whose input-output fails to match. Example tests are the only legitimate correctness signal at the filter stage; hidden tests are invisible here. The goal of this step is to use a cheap signal to compress a huge candidate set to a manageable size.


In [ ]:
def run_tests(fn, tests):
    """Run a program on a list of (input, expected output); return (passed count, total count)."""
    ok = 0
    for x, want in tests:
        try:
            if bool(fn(x)) == want:
                ok += 1
        except Exception:
            pass
    return ok, len(tests)


def filter_by_example_tests(candidates, tests):
    """Return candidates that pass every example test."""
    return [c for c in candidates if run_tests(c["fn"], tests)[0] == len(tests)]


passed = filter_by_example_tests(cands, EXAMPLE_TESTS)
n_passed_good = sum(c["good"] for c in passed)
print("before filter %d candidates -> %d passed example tests (dropped %.1f%%)"
      % (len(cands), len(passed), 100.0 * (1 - len(passed) / len(cands))))
print("among those that passed example tests, %d correct, %d incorrect mixed in"
      % (n_passed_good, len(passed) - n_passed_good))
print("Key observation: example tests do not cover everything, so two wrong variants slip through.")
print("In the paper setting, example tests drop about 99%; this toy task is simpler and has fewer error modes, so the fraction is lower.")


Filtering is the first gate of a cheap signal. Of 300 candidates only 149 pass all 4 example tests; about half are dropped.

Of the remaining 149, 61 are correct and 88 are wrong. Wrong solutions slip through because, of the six error variants, wrong_all_composite, wrong_trial_range, wrong_mod_division, and wrong_stop_early are caught as soon as they hit example tests; only wrong_even_composite and wrong_one_is_prime have defects that fall on inputs the examples do not cover. About one third of the sampled wrong solutions therefore penetrate the filter, so their count exceeds the correct ones. A stronger model has a higher share of correct solutions, so after filtering the correct ones dominate; in the paper setting, example tests drop about 99% of candidates.

The third step is clustering. After filtering, remaining candidates may still exceed the submission quota, and they still mix in wrong solutions that example tests did not catch. Clustering cannot use hidden tests; it uses an extra set of inputs to observe candidate behavior. That set is called **probe inputs**; from it we build behavior signatures and group candidates with the same behavior into a cluster. Correct solutions behave similarly and form one large cluster; wrong solutions fail in different ways and scatter into many small clusters. Representatives are taken starting from the largest cluster, one per cluster, covering as many different solutions as possible.

In [ ]:
from collections import defaultdict


def behavior_signature(c, probe_inputs):
    """The candidate's output tuple on probe_inputs, used as a behavior signature."""
    return tuple(bool(c["fn"](x)) for x in probe_inputs)


def cluster_by_signature(candidates, probe_inputs):
    """Group by behavior signature; return clusters sorted by size descending, each (signature, [candidates])."""
    groups = defaultdict(list)
    for c in candidates:
        groups[behavior_signature(c, probe_inputs)].append(c)
    return sorted(groups.items(), key=lambda kv: len(kv[1]), reverse=True)


def submit_top_clusters(clusters, n_submit=10):
    """Take one representative from each of the largest clusters, as the submission."""
    return [members[0] for _, members in clusters[:n_submit]]


def solves_hidden(c, tests):
    """Whether the candidate passes every hidden test, i.e. whether it solves the problem."""
    return run_tests(c["fn"], tests)[0] == len(tests)


# Test-input generation: boundary values plus random integers, simulating AlphaCode's test-input generation model
probe_inputs = [0, 1, 2, 3, 4, 9, 16, 25, 100] + list(rng.integers(1, 300, size=20))
clusters = cluster_by_signature(passed, probe_inputs)
subs = submit_top_clusters(clusters)
n_solved = sum(solves_hidden(c, HIDDEN_TESTS) for c in subs)

print("after filtering, %d candidates clustered into %d behavior clusters" % (len(passed), len(clusters)))
for i, (sig, members) in enumerate(clusters[:5]):
    n_g = sum(c["good"] for c in members)
    print("  cluster %d size %3d, of which %d correct" % (i, len(members), n_g))
print("submitted %d representatives, of which %d actually solve the hidden tests" % (len(subs), n_solved))
print("summary: sample %d -> filter leaves %d (%.1f%%) -> %d clusters -> submit %d"
      % (len(cands), len(passed), 100.0 * len(passed) / len(cands),
         len(clusters), len(subs)))
print("Key observation: correct solutions form the largest cluster, so clustered submission solves the hidden tests.")


Clustering compresses 149 candidates into 3 behavior clusters; each of the three numbers has a meaning.

Cluster 0 has size 61, all correct solutions. The 3 correct implementations are written differently, but their outputs on probe inputs match, so they share one signature: a correct solution outputs the right answer on any input, so behavior agrees. Cluster 1 has size 50, all wrong_even_composite; this error always misclassifies n = 2, and the probe inputs include 2, so they match each other and differ from the correct solution. Cluster 2 has size 38, all wrong_one_is_prime; the error concentrates on n = 0, 1, so they form their own cluster.

88 wrong solutions are compressed into 2 clusters. At submission each cluster sends one representative: cluster 0's representative solves the hidden tests; the other two representatives carry their defects in the signature and fail as soon as hidden tests run; only 1 representative solves the problem. If two candidates behave the same, submitting one has the same effect as submitting a hundred—that is why clustering saves submission quota.

A single problem is only a snapshot. To see whether more samples buy more solves, we repeat the experiment while changing the sample count. Under a limited submission quota, pipeline quality is measured by two quantities. First the theoretical ceiling: if the quota were unlimited and all k sampled candidates were submitted, the probability of solving the problem is called `pass@k`. Then what can actually be done: the probability of solving after filter and cluster when only 10 may be submitted, called `10@k`. The gap between them is what the selection algorithm loses. The AlphaCode paper reports a further finding: solve rate grows approximately log-linearly with the number of samples, and a stronger model reaches the same solve rate with fewer samples.

Below we expand the task to four toy problems and measure random submission with no pipeline, filter-plus-cluster 10@k, and the analytic pass@k upper bound. Each problem has its own candidate pool, example tests, and hidden tests; probe inputs for behavior signatures are generated in a fixed way.

### pass@k: the theoretical upper bound of submitting everything

This subsection derives the closed form of pass@k. Given per-sample correctness p, and sampling k candidates, it is the highest fraction of problems that can be solved in theory. The same closed form is the reference upper bound for the experiment below.

Suppose each candidate is correct with probability p, and draws are independent. Submit all k candidates; the problem is solved if at least one is correct. The probability that all are wrong is $(1-p)^k$, so

$$ pass@k = 1 - (1-p)^k $$

Substitute p = 0.08 at a few values of k:

| k | $(1-p)^k$ | pass@k |
|:---|:---|:---|
| 10 | 0.92^10 ≈ 0.434 | 0.566 |
| 100 | 0.92^100 ≈ 0.0002 | ≈ 0.9998 |
| 1000 | 0.92^1000 ≈ 6×10^(-37) | ≈ 1.0000 |

Failure probability is an exponential function of k and falls very fast: when samples go from 10 to 100, the failure rate drops from 43% to 0.02%. pass@k is the theoretical upper bound of "submit every candidate, with no selection cost". No algorithm that may submit only 10 can exceed it; the gap between 10@k and pass@k measures what the selection algorithm loses.

### Why solve rate is approximately log-linear in the number of samples

This subsection explains a pattern in the experiment: solve rate grows approximately log-linearly with the number of samples. Log-linear means that each time the sample count is multiplied by a fixed factor, solve rate rises by about a fixed amount; on semi-log paper the curve is approximately a straight line.

Write the failure rate as $f(k)=(1-p)^k$. Taking logs, $\ln f(k) = k\ln(1-p)$, so the log of the failure rate falls linearly in k. Solving for k: to divide the failure rate by 10, the same number of additional samples is required whether the current count is 100 or 1000.

In other words, the sample count has to grow in the multiplicative direction for the failure rate to fall in the dividing direction; the solve rate $1-f(k)$ therefore rises approximately linearly in $\log k$ in the middle of the curve. The paper observed this regularity and summarized it as "solve rate grows approximately log-linearly with the number of samples"; the figure's horizontal axis is accordingly a log scale.

### How many samples are needed to reach a 50% solve rate

A concrete number matters in practice: to reach a solve rate above one half, how many candidates must be sampled. Solving $pass@k = 0.5$ gives $k = \ln 0.5 / \ln(1-p)$:

| Per-sample correctness p | Samples needed for 50% solve rate |
|:---|:---|
| 0.2 | 3 |
| 0.02 | 34 |
| 0.002 | 346 |

Each time p shrinks by 10×, the required sample count grows by about 10×. That is why model quality decides sample efficiency: raising per-sample correctness from 0.2% to 2% drops the samples needed for the same solve rate from hundreds to tens.

In [ ]:
def make_pool(goods, bads):
    """Pack correct variants and wrong variants into a candidate pool."""
    pool = [{"name": "g%d" % i, "fn": f, "good": True} for i, f in enumerate(goods)]
    pool += [{"name": "b%d" % i, "fn": f, "good": False} for i, f in enumerate(bads)]
    return pool


def pow_loop(x):
    """Power of two: repeatedly divide by 2 until 1."""
    if x <= 0:
        return False
    while x > 1:
        if x % 2 == 1:
            return False
        x //= 2
    return True


# Three auxiliary problems use lambda pools; the main demo problem is_prime reuses POOL above.
# Each problem deliberately keeps one wrong variant that example tests miss and hidden tests catch.
PROBLEMS = [
    {"name": "is_prime",
     "pool": POOL,
     "examples": EXAMPLE_TESTS,
     "hidden": HIDDEN_TESTS},
    {"name": "is_square",
     "pool": make_pool(
         [lambda x: int(x ** 0.5) ** 2 == x,
          lambda x: any(i * i == x for i in range(x + 1))],
         [lambda x: any(i * i == x for i in range(x)),
          lambda x: int(x ** 0.5) ** 2 == x and x > 1]),
     "examples": [(1, True), (4, True), (9, True), (8, False)],
     "hidden": [(0, True), (16, True), (2, False), (25, True),
                (3, False), (100, True), (7, False)]},
    {"name": "is_power_of_two",
     "pool": make_pool(
         [lambda x: x > 0 and (x & (x - 1)) == 0, pow_loop],
         [lambda x: (x & (x - 1)) == 0,
          lambda x: x > 1 and (x & (x - 1)) == 0]),
     "examples": [(1, True), (2, True), (4, True), (6, False), (8, True)],
     "hidden": [(0, False), (1, True), (16, True), (3, False), (32, True),
                (10, False), (64, True), (5, False)]},
    {"name": "is_palindrome",
     "pool": make_pool(
         [lambda x: str(x) == str(x)[::-1],
          lambda x: all(str(x)[i] == str(x)[-1 - i]
                        for i in range(len(str(x)) // 2))],
         [lambda x: str(x) == str(x)[::-1] or x > 999,
          lambda x: str(x) == str(x)[::-1] and len(str(x)) % 2 == 1]),
     "examples": [(0, True), (121, True), (123, False), (12321, True), (10, False)],
     "hidden": [(1, True), (22, True), (123, False), (1221, True),
                (100, False), (345, False), (1231, False)]},
]

# Each problem uses probe inputs that separate correct from wrong variants, so a wrong solution does not share a cluster with a correct one
base_probes = [0, 1, 2, 3, 4, 9, 16, 25, 100]
extra_probes = {
    "is_prime": [2, 4, 9, 16, 25, 100],
    "is_square": [0, 1, 4, 9, 100],
    "is_power_of_two": [0, 1, 2, 3, 8, 32],
    "is_palindrome": [11, 22, 121, 123, 1231],
}
for prob in PROBLEMS:
    prob["probes"] = (base_probes + extra_probes[prob["name"]]
                      + list(rng.integers(5, 400, size=12)))

print("built %d toy problems: %s" % (len(PROBLEMS), ", ".join(p["name"] for p in PROBLEMS)))
for prob in PROBLEMS:
    n_g = sum(p["good"] for p in prob["pool"])
    print("  %-15s %d candidates (%d correct)" % (prob["name"], len(prob["pool"]), n_g))


In [ ]:
def sample_pool(rng, problem, n, p_correct):
    """Sample n candidates from problem's pool; p_correct is the probability of drawing a correct one."""
    good = [p for p in problem["pool"] if p["good"]]
    bad = [p for p in problem["pool"] if not p["good"]]
    out = []
    for _ in range(n):
        out.append(rng.choice(good) if rng.random() < p_correct else rng.choice(bad))
    return out


def simulate_pipeline(problem, rng, n_samples, p_correct, n_submit=10):
    """Sample for one problem and run the full pipeline; return whether it was solved."""
    cands = sample_pool(rng, problem, n_samples, p_correct)
    passed = filter_by_example_tests(cands, problem["examples"])
    if not passed:
        return False
    clusters = cluster_by_signature(passed, problem["probes"])
    subs = submit_top_clusters(clusters, n_submit)
    return any(solves_hidden(c, problem["hidden"]) for c in subs)


def solve_rate(problems, rng, k, p_correct, n_trials=15):
    """Repeat the experiment on several problems; return the mean solve rate."""
    hits = 0
    total = 0
    for prob in problems:
        for _ in range(n_trials):
            hits += simulate_pipeline(prob, rng, k, p_correct)
            total += 1
    return hits / total


print("sample_pool / simulate_pipeline / solve_rate defined")
print("solve rate at k=100, p=0.2: %.3f"
      % solve_rate(PROBLEMS, rng, 100, 0.2, n_trials=15))


In [ ]:
import matplotlib.pyplot as plt

ks = np.array([10, 30, 100, 300, 1000], dtype=float)
pc = 0.08


def solve_rate_raw(problems, rng, k, p_correct, n_trials=20):
    """Pick 10 at random from all samples (no filter or cluster); return the solve rate."""
    hits = 0
    total = 0
    for prob in problems:
        for _ in range(n_trials):
            cands = sample_pool(rng, prob, k, p_correct)
            rng.shuffle(cands)
            subs = cands[:10]
            hits += any(solves_hidden(c, prob["hidden"]) for c in subs)
            total += 1
    return hits / total


# Left: random submission with no pipeline vs filter plus cluster, and the pass@k upper bound
y_raw = np.array([solve_rate_raw(PROBLEMS, rng, int(k), pc) for k in ks])
y_pipe = np.array([solve_rate(PROBLEMS, rng, int(k), pc) for k in ks])
ypass = 1.0 - (1.0 - pc) ** ks

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ks, y_raw, marker="o", label="random (no pipeline)")
ax1.plot(ks, y_pipe, marker="s", label="filter+cluster (10@k)")
ax1.plot(ks, ypass, marker="^", ls="--", label="pass@k (upper bound)")
ax1.set_xscale("log")
ax1.set_xlabel("number of samples k (log scale)")
ax1.set_ylabel("solve rate")
ax1.set_title("Pipeline vs raw sampling")
ax1.legend()

# Right: analytic log-linear growth of pass@k. Real contest problems have very low per-sample correctness p.
for p in [0.002, 0.005, 0.02]:
    y = 1.0 - (1.0 - p) ** ks
    ax2.plot(ks, y, marker="o", label="p=%.3f" % p)
    slope = np.polyfit(np.log10(ks), y, 1)[0]
    print("p=%.3f log-linear fitted slope: %.2f" % (p, slope))
ax2.set_xscale("log")
ax2.set_xlabel("number of samples k (log scale)")
ax2.set_ylabel("solve rate")
ax2.set_title("Log-linear growth of pass@k")
ax2.legend()

plt.tight_layout()
plt.show()
for p in [0.002, 0.02, 0.2]:
    k_half = np.log(0.5) / np.log(1 - p)
    print("samples needed for 50%% solve rate: p=%.3f -> %.0f" % (p, k_half))
print("Key observation: random submission's solve rate is capped by per-sample correctness; filter plus cluster approaches the upper bound;")
print("higher p shifts the log-linear curve left, so fewer samples are needed for the same solve rate.")


In the previous section's pipeline, which representative to pick inside a cluster was arbitrary. This section picks a more reliable representative inside a cluster. The method is to estimate each candidate's correctness with a scoring model inside the cluster and submit the highest-scoring one. That action is **reranking**: sort candidates by score and take the one in front.

AlphaCode 2 keeps the sample, filter, and cluster kit, only upgrading "pick one per cluster" to "pick the best per cluster", then swapping in a stronger base model; sample efficiency improves by more than 10,000×. Below we watch the value of reranking in two ways, starting with a comparison of four submission strategies: pick at random, filter only, filter plus cluster, and filter plus cluster plus in-cluster scoring.

Reranking changes only the "pick one per cluster" step: first select candidate clusters by cluster size, then inside a cluster estimate each candidate's correctness with a scoring model and submit the highest-scoring one. The change is small in scope and large in effect.

Comparing four strategies shows each layer's contribution. Random pick uses no signal and is chance. Filter only uses example tests to raise the density of correct candidates. Filter plus cluster splits candidates by behavior, so 10 submissions cover different solutions. Filter plus cluster plus in-cluster scoring further drops individuals that behave the same but have a defective implementation, and picks the most reliable representative.

score_by_simulated_model uses the good flag to simulate a scoring model's 0-1 estimate. In a real system this score is predicted by a fine-tuned model on a subset of hidden tests; here whether the candidate is actually correct stands in as an idealized version, to demonstrate the value of scoring. Below we run 40 trials on each of 4 problems and compare the four strategies' solve rates.

In [ ]:
def score_by_simulated_model(c):
    """Simulate a fine-tuned scoring model: a 0-1 estimate of candidate correctness (using the good flag)."""
    return 1.0 if c["good"] else 0.0


def strategy_compare(problem, rng, n_samples, p_correct, n_submit=10):
    """Compare four submission strategies; return each strategy's solve bit (0/1).

    Four strategies: pick at random from all samples, pick at random after filtering,
    pick representatives with filter plus cluster,
    filter plus cluster plus in-cluster scoring.
    """
    cands = sample_pool(rng, problem, n_samples, p_correct)
    passed = filter_by_example_tests(cands, problem["examples"])
    if not passed:
        return (0, 0, 0, 0)
    shuffled = cands[:]
    rng.shuffle(shuffled)
    random_subs = shuffled[:n_submit]
    filter_subs = passed[:n_submit]
    clusters = cluster_by_signature(passed, problem["probes"])
    cluster_subs = submit_top_clusters(clusters, n_submit)
    scored_subs = [max(members, key=score_by_simulated_model)
                   for _, members in clusters[:n_submit]]
    hid = problem["hidden"]
    solved = lambda subs: any(solves_hidden(c, hid) for c in subs)
    return (solved(random_subs), solved(filter_subs),
            solved(cluster_subs), solved(scored_subs))


agg = {"random": [], "filter": [], "cluster": [], "cluster+score": []}
for prob in PROBLEMS:
    for _ in range(40):
        r_, f_, c_, cs_ = strategy_compare(prob, rng, 200, 0.08)
        agg["random"].append(r_)
        agg["filter"].append(f_)
        agg["cluster"].append(c_)
        agg["cluster+score"].append(cs_)

print("solve rates of four submission strategies under weak quality (p_correct=0.08) (4 problems × 40 trials):")
for k, v in agg.items():
    print("  %-14s %.2f" % (k, np.mean(v)))
print("Key observation: random submission with no pipeline has the lowest hit rate; filtering raises correct-candidate density;")
print("clustering covers different behavior clusters, so submitting the cluster that holds a correct solution solves the problem.")


In [ ]:
# Weak-model setting: when a correct solution is not the largest cluster, clustering fails and a scoring model compensates
rng2 = np.random.default_rng(7)
weak = filter_by_example_tests(
    sample_candidates(rng2, 400, p_correct=0.08), EXAMPLE_TESTS)
weak_clusters = cluster_by_signature(weak, probe_inputs)
weak_cluster_subs = submit_top_clusters(weak_clusters)
weak_score_subs = sorted(weak, key=score_by_simulated_model, reverse=True)[:10]

print("under a weak model (p_correct=0.08), filtering leaves %d candidates" % len(weak))
print("largest cluster size %d, of which %d correct"
      % (len(weak_clusters[0][1]), sum(c["good"] for c in weak_clusters[0][1])))
print("cluster submission solves: %d, scored submission solves: %d"
      % (sum(solves_hidden(c, HIDDEN_TESTS) for c in weak_cluster_subs),
         sum(solves_hidden(c, HIDDEN_TESTS) for c in weak_score_subs)))
print("Key observation: when correct samples are few, a wrong solution's behavior cluster is larger, "
      "so picking by cluster size misses the correct solution; picking by a correctness estimate does not.")


The four-strategy comparison assumed that correct solutions form the largest cluster; the weak-model setting removes that assumption. At p_correct = 0.08, about 32 of 400 samples are correct; after filtering, 150 candidates remain, and the largest cluster of 58 contains no correct solution, because wrong_even_composite has a single defect, so those programs behave alike and form a larger cluster.

Picking by cluster size takes that large cluster's representative first; the small cluster that holds a correct solution ranks second; 10 slots bring in only 1 correct representative, so only 1 problem is solved. A scoring model ignores cluster size and sorts by estimated correctness, taking the top 10; correct solutions all rank at the front, and all 10 submissions solve the problem.

The contrast shows that cluster size reflects how often this writing was sampled, not whether this writing is correct. When model quality is low and correct samples are scarce, cluster size is an unreliable ranking signal, and a correctness signal is needed as a backstop.

The first two sections searched in program space. This section moves to knowledge space and treats another problem: a long-reasoning model is halfway through a chain and finds that it lacks a fact. The model's memory has a boundary; knowledge not seen in training cannot be invented correctly, so it has to look up material on demand during inference.

To handle a knowledge gap, we first have to observe it. Statistics show that when a long-reasoning model is unsure it tends to emit hedge words rather than invent an answer; for example perhaps appears on average 30 times per output. Those words are a signal of a knowledge gap: at some step the model finds it lacks a fact. The next cell first builds a reasoning chain and counts the uncertainty words in it.

In [ ]:
# Hand-built long reasoning text, simulating uncertainty words that o1-style models emit at high frequency
chain = (
    "This problem is physics. wait, let me think. perhaps relativity applies here, "
    "maybe I have the formula backward. likely I need the Lorentz factor, "
    "hmm I am unsure about this section, wait I could ignore it for now. "
    "perhaps I should first assume a rest frame."
)
tokens = ["perhaps", "wait", "maybe", "likely", "hmm"]
counts = {t: chain.count(t) for t in tokens}
print("uncertainty-word counts in the reasoning chain:")
for t, c in counts.items():
    print("  %-8s %d" % (t, c))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.bar(list(counts.keys()), list(counts.values()), color="#4C72B0")
plt.xlabel("uncertainty word")
plt.ylabel("count")
plt.title("Uncertainty markers in a reasoning chain")
plt.tight_layout()
plt.show()
print("Key observation: these uncertainty words are a signal of a knowledge gap in reasoning; Search-o1 treats them as cues to trigger retrieval.")


When a long-reasoning model is unsure it tends to emit hedge words rather than invent an answer, so the frequency of words such as perhaps and maybe becomes an observation window on reasoning confidence. The paper's statistics show that such uncertainty words appear at high frequency in long-reasoning outputs; for example perhaps appears on average 30 times per output. The denser these words are in a reasoning chain, the closer the model is to a knowledge boundary in that stretch.

In the hand-built chain, perhaps, wait, maybe, likely, and hmm each appear 1 or 2 times. The bar chart plots those counts and shows uncertainty words concentrating where the model is unfamiliar with a stretch. Search-o1's idea is: rather than let the model keep inventing vaguely, treat those positions as retrieval triggers, pause, look up, then continue.

Search-o1 splits "handling a knowledge gap" into two steps: when to search, and how to use what is found, each given to a component.

When to search is left to the **reasoning model itself**. At an uncertain point it emits a search query wrapped in special symbols; detecting the query-end marker pauses reasoning and fires retrieval. How to use the result is left to a **Reason-in-Documents module** (the paper's name). It runs independently of the main reasoning chain: first it condenses retrieved documents into a short piece of knowledge that can be used directly in reasoning, then it splices that piece back into the chain with result symbols, and reasoning continues. The two steps run one after the other, so one reasoning chain can trigger several retrieval rounds, covering different knowledge needs at different steps.

The next cells reproduce this loop with a small knowledge base and the unified LLM client.

In [ ]:
# Unified LLM client: use a real model when an API key is set, otherwise fall through to the scripted path
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()
print("LLM mode:", "scripted example (deterministic placeholder)" if False else "real")

# Small knowledge base: 20 facts; keys are keywords, values are citable content
KB = {
    "Einstein": "Einstein was born in Ulm, Germany, and was a theoretical physicist.",
    "Ulm": "Ulm is a city in Baden-Wurttemberg, Germany, on the Danube.",
    "Germany": "Germany is a European country. Its capital is Berlin.",
    "Berlin": "Berlin is the capital of Germany, with a population of about 3.6 million.",
    "France": "France is a European country. Its capital is Paris.",
    "Paris": "Paris is the capital of France, with a population of about 2.1 million.",
    "Newton": "Newton was a British physicist who formulated the law of universal gravitation.",
    "gravity": "The law of universal gravitation describes attraction between masses.",
    "relativity": "Relativity is the theory of spacetime proposed by Einstein.",
    "lightspeed": "The speed of light is about 300,000 kilometers per second.",
    "massenergy": "The mass-energy equation E=mc^2 comes from Einstein.",
    "China": "China is an Asian country. Its capital is Beijing.",
    "Beijing": "Beijing is the capital of China.",
    "Japan": "Japan is an Asian country. Its capital is Tokyo.",
    "Tokyo": "Tokyo is the capital of Japan, with a population of about 14 million.",
    "Britain": "Britain is a European country. Its capital is London.",
    "London": "London is the capital of Britain, with a population of about 9 million.",
    "Linus": "Linus created the Linux kernel and was born in Finland.",
    "Finland": "Finland is a Nordic country. Its capital is Helsinki.",
    "Helsinki": "Helsinki is the capital of Finland.",
}


def retrieve(query, kb):
    """Retrieve from the knowledge base by keyword; return a list of matching (key, value)."""
    tokens = query.split()
    hits = []
    for key, value in kb.items():
        if any(tok in key for tok in tokens):
            hits.append((key, value))
    return hits


print("retrieve example 'Germany Berlin':", retrieve("Germany Berlin", KB))


In [ ]:
SEARCH_OPEN = "<|begin_search_query|>"
SEARCH_CLOSE = "<|end_search_query|>"
RESULT_OPEN = "<|begin_search_result|>"
RESULT_CLOSE = "<|end_search_result|>"


def extract_queries(text):
    """Return search queries in the text, in order of appearance."""
    queries = []
    parts = text.split(SEARCH_OPEN)
    for part in parts[1:]:
        if SEARCH_CLOSE in part:
            queries.append(part.split(SEARCH_CLOSE)[0])
    return queries


def make_reasoner(client):
    """Build Search-o1's reasoning-actor: emit a reasoning chain and trigger search queries.

    On the scripted path, return a scripted trace; in real mode, call the underlying model. Return (first, next_):
    first generates the start of the chain, next_ continues after a retrieval result is injected.
    """
    if False:
        steps = [
            "I remember Einstein was born in Germany, but I am unsure of the city."
            "<|begin_search_query|>Einstein birthplace<|end_search_query|>",
            "Ulm is in Germany. Next comes Germany's capital."
            "<|begin_search_query|>Germany capital<|end_search_query|>",
            "Combining birthplace and capital, the answer should be Berlin.",
        ]
        state = {"i": 0}

        def first(question):
            return steps[0]

        def next_(context):
            state["i"] += 1
            return steps[min(state["i"], len(steps) - 1)]

        return first, next_

    def first(question):
        return client.chat([{"role": "user",
                             "content": question + " Reason step by step; at uncertain points use "
                             + SEARCH_OPEN + "query" + SEARCH_CLOSE + " to trigger retrieval."}])

    def next_(context):
        return client.chat([{"role": "user",
                             "content": "Continue reasoning: " + context}])

    return first, next_


def refine_docs(chain, query, docs, client):
    """retrieval-critic: refine retrieved documents into knowledge that can be injected into the reasoning chain."""
    if False:
        if docs:
            return "According to the sources: " + docs[0][1]
        return "No relevant information retrieved."
    prompt = ("Current reasoning chain: %s\nsearch query: %s\nretrieved documents: %s\n"
              "Condense the documents into concise knowledge that can be used directly in reasoning."
              % (chain, query, docs))
    return client.chat([{"role": "user", "content": prompt}])


In [ ]:
def run_search_o1(question, kb, client, max_rounds=3):
    """Run a Search-o1-style retrieval-injection loop; return (final chain, retrieval rounds, query list)."""
    first, next_ = make_reasoner(client)
    chain = first(question)
    rounds = 0
    processed = 0
    queries_done = []
    while rounds < max_rounds:
        queries = extract_queries(chain)
        if len(queries) <= processed:
            break
        query = queries[processed]
        processed += 1
        docs = retrieve(query, kb)
        refined = refine_docs(chain, query, docs, client)
        chain = chain + RESULT_OPEN + refined + RESULT_CLOSE
        chain = chain + next_(chain)
        queries_done.append(query)
        rounds += 1
    return chain, rounds, queries_done


question = "Einstein was born in which country? What is the capital of that country?"
final_chain, n_rounds, queries = run_search_o1(question, KB, client)
print("retrieval rounds: %d" % n_rounds)
for i, q in enumerate(queries):
    print("  round %d query: %s" % (i + 1, q))
print("final reasoning chain:")
print(final_chain)
print("Key observation: one reasoning chain triggers two retrieval rounds, covering different knowledge needs at each step.")


Walk through one retrieval injection on the scripted trace. In round 1 the reasoning chain pauses at "I am unsure of the city" and then writes a query wrapped in symbols: `<|begin_search_query|>Einstein birthplace<|end_search_query|>`. extract_queries splits on SEARCH_OPEN and takes the part before SEARCH_CLOSE, extracting the query text. retrieve keyword-matches in the knowledge base and hits the Einstein entry.

The Reason-in-Documents module condenses the hit into one sentence, "Einstein was born in Ulm, Germany, and was a theoretical physicist", then splices it onto the end of the chain with result symbols `<|begin_search_result|>...<|end_search_result|>`. With that knowledge attached, the chain continues to generate the next stretch. Round 2 is the same: the query "Germany capital" hits "Germany is a European country. Its capital is Berlin.", and after injection the model gives the final answer "Berlin".

The two components each have one job: the reasoning model only decides when to search and what to search, and does not worry about how to use the result; the Reason-in-Documents module only condenses retrieved documents into usable knowledge and splices it back, running independently of the main chain and not interrupting reasoning. One reasoning chain can therefore trigger several retrieval rounds, covering different knowledge needs at different steps.

The previous section's Search-o1 lets the model decide when to search. Another, more common method is: first look up material from the question, stuff the hits into the prompt, then let the model answer. That method is retrieval-augmented generation, abbreviated **RAG**. The most naive version, `standard RAG`, retrieves once from the original question.

One retrieval is not enough for many questions. Some questions take several steps to answer: first obtain an intermediate fact, then follow it to the final answer. Those questions are **multi-hop**. The intermediate entity in a multi-hop question often does not appear in the original question, so one retrieval cannot cover different knowledge gaps at each step. On-demand retrieval (`agentic RAG`) fires a query at each step and can compose an answer that spans several entities. The paper reports that on multi-hop open-domain QA, on-demand retrieval raises mean EM (exact match: the answer matches the gold answer word for word) by 23.2% over standard RAG.

In [ ]:
def standard_rag(question, kb, client):
    """Standard RAG: retrieve once from the original question, then generate the answer directly."""
    docs = retrieve(question, kb)
    if False:
        if not docs:
            msg = "scripted example: retrieving from the original question hit no entries, so multi-hop is not covered."
        else:
            msg = "scripted example: hit %d entries, intermediate entities not covered." % len(docs)
        return msg, docs
    prompt = "Answer the question from the following sources: %s\nQuestion: %s" % (docs, question)
    return client.chat([{"role": "user", "content": prompt}]), docs


std_answer, std_docs = standard_rag(question, KB, client)
_, n_agent_rounds, agent_queries = run_search_o1(question, KB, client)
print("Standard RAG: 1 query, %d hits" % len(std_docs))
print("  " + std_answer)
print("Agentic RAG: %d retrieval rounds, queries in order: %s"
      % (n_agent_rounds, " -> ".join(agent_queries)))
print("Key observation: a multi-hop question's intermediate entity is not in the original question, "
      "so one retrieval cannot cover it; on-demand retrieval is what assembles a complete answer.")


Standard RAG retrieves once from the original question. The question "Einstein was born in which country? What is the capital of that country?" depends on two facts: Einstein was born in Germany (first hop), and Germany's capital is Berlin (second hop). The second hop's key entity "Germany" does not appear in the original question, so retrieval cannot take it as a target.

Naive token overlap on the raw question can hit the Einstein entry (the word Einstein is in the question) but does not hit Germany, because that name is not in the query. On-demand retrieval hands "when to search, what to search" to the reasoning chain: round 1 extracts "Einstein birthplace", round 2 extracts "Germany capital", the two hops hit separately and are joined into a complete answer. The paper reports that on multi-hop open-domain QA, on-demand retrieval raises mean EM (exact match) by 23.2% over standard RAG.

The components of the first three sections each solve one problem. This section chains them to answer a larger one: how to have a model answer an open research question, such as "compare two German cities in status and population". This multi-step workflow of decompose, then retrieve, then write is called **deep research**. The same-named feature in products such as ChatGPT does this as well.

Deep research faces an open question and has four stages: decompose a complex question into several retrievable subquestions (planning), run retrieval on each subquestion (agentic RAG), synthesize evidence into a report (synthesis), and attach sources to the report (citation). AlphaCode's filter-plus-cluster handles candidate programs, Search-o1's retrieval injection handles knowledge entries, and the deep-research workflow chains them into a pipeline aimed at one research question.

Deep research splits "answer an open question" into four stages, each corresponding to a component already seen.

The planning stage decomposes the topic into retrievable subquestions. The topic "German cities: differences in status and population between Ulm and Berlin" is split into four queries: Ulm city, Berlin, Ulm Germany, Berlin population. Each query focuses on one independent fact, the same idea as Search-o1 retrieving one knowledge gap per round. The per-subquestion retrieval stage runs agentic RAG once per subquestion; here that is four independent retrieval rounds, rather than chaining them on one reasoning chain as Search-o1 does.

The synthesis stage merges each subquestion's evidence into a report. The scripted path lists evidence entries directly; real mode has the model organize evidence into a coherent narrative. The citation stage records which knowledge-base entry each piece of evidence came from, so every claim in the report can be traced to a source.

The relation to earlier components: planning and on-demand retrieval reuse Search-o1's "split retrieval small" idea; filtering and refining reuse AlphaCode and Search-o1's Reason-in-Documents module's "keep only usable information" idea. The pipeline introduces no new mechanism; it reorganizes existing components for a research setting.

In [ ]:
def plan_questions(topic, client):
    """Decompose a research topic into retrievable subquestions (planning stage)."""
    if False:
        return ["Ulm city", "Berlin", "Ulm Germany", "Berlin population"]
    content = "Decompose the topic '%s' into 3-4 retrievable subquestions." % topic
    return [line.strip() for line in client.chat([{"role": "user",
                                                   "content": content}]).splitlines()]


def synthesize_report(topic, evidence, client):
    """Synthesize each subquestion's retrieved evidence into a report (synthesis stage)."""
    if False:
        lines = ["# Topic: " + topic]
        for q, docs in evidence:
            for key, val in docs[:1]:
                lines.append("- %s: %s" % (key, val))
        lines.append("Conclusion: the entries above come from the knowledge base; in a real setting they would be synthesized from retrieval results.")
        return "\n".join(lines)
    prompt = "Write a concise report from the following evidence: %s" % (evidence,)
    return client.chat([{"role": "user", "content": prompt}])


def build_references(evidence):
    """Build a citation list from evidence (citation stage)."""
    refs = []
    for q, docs in evidence:
        for key, val in docs:
            refs.append("%s -- %s" % (key, val))
    return refs


def deep_research(topic, kb, client):
    """Mini deep research: plan -> retrieve per subquestion -> synthesize -> cite."""
    questions = plan_questions(topic, client)
    evidence = []
    for q in questions:
        evidence.append((q, retrieve(q, kb)))
    report = synthesize_report(topic, evidence, client)
    refs = build_references(evidence)
    return report, refs


In [ ]:
topic = "German cities: differences in status and population between Ulm and Berlin"
report, refs = deep_research(topic, KB, client)
print(report)
print("%d citations:" % len(refs))
for r in refs[:8]:
    print("  [%s]" % r)
print("(On the scripted path the report is a placeholder; with an API key configured, a real model synthesizes it.)")


## Summary

- Program synthesis is search in a huge program space; the sample, filter, and cluster kit compresses a huge candidate set into a limited submission
- Example tests are a cheap verifier and drop most candidates; hidden tests cannot be used at the filter stage
- A behavior signature groups candidates with the same output; correct solutions form a large cluster, wrong solutions scatter
- pass@k is the upper bound of submitting every candidate; the gap between 10@k and pass@k measures what the selection algorithm loses
- Solve rate grows approximately log-linearly with the number of samples; higher model quality makes the curve rise earlier and needs fewer samples
- AlphaCode 2 picks the best inside a cluster with a scoring model and swaps in a stronger base model; sample efficiency improves by more than 10,000×
- Search-o1 lets the reasoning model decide when to retrieve; the Reason-in-Documents module refines retrieval results
- One retrieval (standard RAG) cannot cover a multi-hop question's intermediate entity; on-demand repeated retrieval is what assembles a complete answer
- A deep-research workflow is the chain of four stages: plan, retrieve per subquestion, synthesize, cite


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.


**Exercise 1: Filter by example tests**

Implement filter_by_example_tests so that example tests compress three candidates down to the correct one. f_ok behaves correctly; the two wrong variants fail on n=1 and n=2 respectively; the example tests already cover those two inputs.

Hint: first run fn(x) on each input to get the output, then compare with the expected value; return those that pass all tests.


In [ ]:
def f_ok(n):
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def f_wrong_one(n):
    if n == 1:
        return True
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def f_wrong_two(n):
    if n % 2 == 0:
        return False
    for i in range(3, n):
        if n % i == 0:
            return False
    return True


candidates = [{"name": "ok", "fn": f_ok},
              {"name": "wrong_one", "fn": f_wrong_one},
              {"name": "wrong_two", "fn": f_wrong_two}]
tests = [(1, False), (2, True), (3, True), (4, False), (9, False)]


def filter_by_example_tests(candidates, tests):
    """Return candidates that pass every example test (fill in for this exercise)."""
    keep = []
    for c in candidates:
        ok = True
        for x, want in tests:
            if bool(c["fn"](x)) != want:
                ok = False
                break
        if ok:
            keep.append(c)
    return keep


survivors = filter_by_example_tests(candidates, tests)
assert {c["name"] for c in survivors} == {"ok"}
assert len(survivors) == 1
print("kept after filtering:", [c["name"] for c in survivors])
print("Passed: example tests compressed three candidates down to the one correct program.")


**Exercise 2: Cluster by behavior signature**

Implement cluster_by_signature. outputs[i] is program i's output vector on a set of probe inputs; group programs with the same vector into a cluster, and return the cluster list sorted by size descending.

Hint: an output tuple can be a dict key; collect indices with the same signature into the same list.


In [ ]:
outputs = [
    [1, 0, 0, 1],   # correct program A
    [1, 0, 0, 1],   # correct program B (same behavior as A)
    [0, 0, 0, 1],   # wrong program C
    [0, 0, 0, 0],   # wrong program D
]


def cluster_by_signature(outputs):
    """Group programs with the same output vector into a cluster; return clusters sorted by size descending.

    Each cluster is a list of program indices (fill in for this exercise).
    """
    groups = {}
    for i, row in enumerate(outputs):
        key = tuple(row)
        groups.setdefault(key, []).append(i)
    return sorted(groups.values(), key=len, reverse=True)


clusters = cluster_by_signature(outputs)
assert len(clusters) == 3
assert clusters[0] == [0, 1]          # the two correct programs form the largest cluster
print("clusters:", clusters)
print("Passed: programs with the same behavior cluster together; the cluster of correct programs is the largest.")


**Exercise 3: Retrieval trigger and result injection**

Implement extract_queries and inject_results. The chain wraps search queries in special symbols; the former extracts queries in order of appearance, the latter splices refined results onto the end of the chain with result symbols.

Hint: first split on SEARCH_OPEN, then take the part of each piece before SEARCH_CLOSE; injection is only string concatenation.


In [ ]:
SEARCH_OPEN = "<|begin_search_query|>"
SEARCH_CLOSE = "<|end_search_query|>"
RESULT_OPEN = "<|begin_search_result|>"
RESULT_CLOSE = "<|end_search_result|>"

chain = ("First recall Newton's work. wait, not sure."
         "<|begin_search_query|>Newton nationality<|end_search_query|>"
         "He was a British scientist. Next, his birthplace."
         "<|begin_search_query|>Newton birthplace<|end_search_query|>")


def extract_queries(text):
    """Return queries in the text in order of appearance (fill in for this exercise)."""
    queries = []
    parts = text.split(SEARCH_OPEN)
    for part in parts[1:]:
        if SEARCH_CLOSE in part:
            queries.append(part.split(SEARCH_CLOSE)[0])
    return queries


def inject_results(chain, query, result):
    """Splice the refined result onto the end of the chain with RESULT_OPEN...RESULT_CLOSE (fill in for this exercise)."""
    return chain + RESULT_OPEN + result + RESULT_CLOSE


queries = extract_queries(chain)
assert queries == ["Newton nationality", "Newton birthplace"]
updated = inject_results(chain, queries[0], "Newton was a British physicist")
assert RESULT_OPEN in updated
assert updated.endswith(RESULT_CLOSE)
print("query list:", queries)
print("chain tail after injection:", updated[-24:])
print("Passed: parsing and injection of the special-symbol pair are both complete.")


## References

- Li et al., [Competition-Level Code Generation with AlphaCode](https://arxiv.org/abs/2203.07814), 2022 — founding system of program synthesis as search; original source of the sample, filter, and cluster kit
- DeepMind, [AlphaCode 2 Technical Report](https://storage.googleapis.com/deepmind-media/AlphaCode2/AlphaCode2_Tech_Report.pdf), 2023 — technical report, no arXiv id; the same system's new reranker and higher sample efficiency
- Luo et al., [Search-o1: Agentic Search-Enhanced Large Reasoning Models](https://arxiv.org/abs/2501.05366), 2025 — on-demand retrieval and document refining for long-reasoning models; source of the Reason-in-Documents module
- DeepMind, [CodeContests dataset](https://github.com/deepmind/code_contests) — AlphaCode's training and evaluation dataset, with time splits and generated tests
- QwQ-32B-Preview, [arXiv:2412.10903](https://arxiv.org/abs/2412.10903) — Search-o1's reasoning backbone, an open long-reasoning model
- Yao et al., [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629), 2022 — source of the agentic idea "think first, then decide to call"
- Asai et al., [Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection](https://arxiv.org/abs/2310.11511), 2023 — judging retrieval necessity and self-reflection, same lineage as Search-o1
- Stanford, [CS329A course outline](https://cs329a.stanford.edu/) — this lecture's place on the course map
